In [12]:
# Environment check
import sys
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("Python:")
print(sys.executable)

print("\nOpenCV:", cv2.__version__)
print("NumPy:", np.__version__)

Python:
C:\Users\nsvin\.conda\envs\vision3d\python.exe

OpenCV: 5.0.0
NumPy: 2.4.6


In [13]:
# Define the dataset paths
RAW_DIR = Path("../data/images")

print("Raw dataset directory:")
print(RAW_DIR.resolve())

print("\nDirectory exists:", RAW_DIR.exists())

Raw dataset directory:
C:\Users\nsvin\DSAI\GerrardHall3D\data\images

Directory exists: True


In [16]:
#Find all image files
IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".JPG",
    ".JPEG",
    ".PNG"
}

all_images = sorted(
    [
        p for p in RAW_DIR.iterdir()
        if p.is_file() and p.suffix in IMAGE_EXTENSIONS
    ]
)

print("Total image files:", len(all_images))

Total image files: 100


In [22]:
#Inspect filenames
for i, path in enumerate(all_images[:30]):
    print(f"{i:3d}  {path.name}")

  0  IMG_2331.JPG
  1  IMG_2332.JPG
  2  IMG_2333.JPG
  3  IMG_2334.JPG
  4  IMG_2335.JPG
  5  IMG_2336.JPG
  6  IMG_2337.JPG
  7  IMG_2338.JPG
  8  IMG_2339.JPG
  9  IMG_2340.JPG
 10  IMG_2341.JPG
 11  IMG_2342.JPG
 12  IMG_2343.JPG
 13  IMG_2344.JPG
 14  IMG_2345.JPG
 15  IMG_2346.JPG
 16  IMG_2347.JPG
 17  IMG_2348.JPG
 18  IMG_2349.JPG
 19  IMG_2350.JPG
 20  IMG_2351.JPG
 21  IMG_2352.JPG
 22  IMG_2353.JPG
 23  IMG_2354.JPG
 24  IMG_2355.JPG
 25  IMG_2356.JPG
 26  IMG_2357.JPG
 27  IMG_2358.JPG
 28  IMG_2359.JPG
 29  IMG_2360.JPG


In [21]:
# Inspect image dimensions
image_info = []

for path in all_images:
    img = cv2.imread(str(path))

    if img is None:
        image_info.append({
            "filename": path.name,
            "width": None,
            "height": None,
            "channels": None,
            "readable": False
        })
    else:
        h, w = img.shape[:2]
        channels = img.shape[2] if img.ndim == 3 else 1

        image_info.append({
            "filename": path.name,
            "width": w,
            "height": h,
            "channels": channels,
            "readable": True
        })

print("Total files:", len(image_info))
print(
    "Readable:",
    sum(x["readable"] for x in image_info)
)
print(
    "Unreadable:",
    sum(not x["readable"] for x in image_info)
)

Total files: 100
Readable: 100
Unreadable: 0


In [26]:
# Detect duplicates
import hashlib

def file_hash(path):
    md5 = hashlib.md5()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            md5.update(chunk)

    return md5.hexdigest()

file_hashes = {}

for path in all_images:
    h = file_hash(path)

    if h not in file_hashes:
        file_hashes[h] = []

    file_hashes[h].append(path)

print("Total files:", len(all_images))
print("Unique file contents:", len(file_hashes))

Total files: 100
Unique file contents: 100


In [29]:
# Inpsect what is in the folder

print("Raw directory:", RAW_DIR.resolve())

print("\nTotal files in directory:")
print(len(list(RAW_DIR.iterdir())))

print("\nImage files:")
print(len(all_images))

print("\nFirst 20 images:")
for i, path in enumerate(all_images[:20]):
    print(i, path.name)
print("\nLast 10 images:")
for i, path in enumerate(all_images[-10:], start=len(all_images)-10):
    print(i, path.name)

Raw directory: C:\Users\nsvin\DSAI\GerrardHall3D\data\images

Total files in directory:
101

Image files:
100

First 20 images:
0 IMG_2331.JPG
1 IMG_2332.JPG
2 IMG_2333.JPG
3 IMG_2334.JPG
4 IMG_2335.JPG
5 IMG_2336.JPG
6 IMG_2337.JPG
7 IMG_2338.JPG
8 IMG_2339.JPG
9 IMG_2340.JPG
10 IMG_2341.JPG
11 IMG_2342.JPG
12 IMG_2343.JPG
13 IMG_2344.JPG
14 IMG_2345.JPG
15 IMG_2346.JPG
16 IMG_2347.JPG
17 IMG_2348.JPG
18 IMG_2349.JPG
19 IMG_2350.JPG

Last 10 images:
90 IMG_2421.JPG
91 IMG_2422.JPG
92 IMG_2423.JPG
93 IMG_2424.JPG
94 IMG_2425.JPG
95 IMG_2426.JPG
96 IMG_2427.JPG
97 IMG_2428.JPG
98 IMG_2429.JPG
99 IMG_2430.JPG


In [30]:
sizes = []

for path in images:
    img = cv2.imread(str(path))
    if img is not None:
        h, w = img.shape[:2]
        sizes.append((w, h))

print("Number successfully loaded:", len(sizes))
print("First 10 dimensions:")

for size in sizes[:10]:
    print(size)

Number successfully loaded: 200
First 10 dimensions:
(5616, 3744)
(5616, 3744)
(5616, 3744)
(5616, 3744)
(5616, 3744)
(5616, 3744)
(5616, 3744)
(5616, 3744)
(5616, 3744)
(5616, 3744)


In [31]:
#check all file extensions
from collections import Counter

extensions = Counter(
    path.suffix.lower()
    for path in RAW_DIR.iterdir()
    if path.is_file()
)

print(extensions)

Counter({'.jpg': 100})


In [32]:
# Working-resolution versions of all 100 images

WORKING_DIR = Path("../data/working_images")
WORKING_DIR.mkdir(parents=True, exist_ok=True)

scale = 0.25

for path in all_images:

    img = cv2.imread(str(path))

    if img is None:
        print("Could not read:", path.name)
        continue

    h, w = img.shape[:2]

    resized = cv2.resize(
        img,
        (int(w * scale), int(h * scale)),
        interpolation=cv2.INTER_AREA
    )

    cv2.imwrite(
        str(WORKING_DIR / path.name),
        resized
    )

print(
    "Working images created:",
    len(list(WORKING_DIR.glob("*")))
)

Working images created: 100


In [33]:
# Verify the working images
working_images = sorted(
    [
        p for p in WORKING_DIR.iterdir()
        if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    ]
)

print("Working image count:", len(working_images))

for i, path in enumerate(working_images[:10]):
    print(i, path.name)

Working image count: 100
0 IMG_2331.JPG
1 IMG_2332.JPG
2 IMG_2333.JPG
3 IMG_2334.JPG
4 IMG_2335.JPG
5 IMG_2336.JPG
6 IMG_2337.JPG
7 IMG_2338.JPG
8 IMG_2339.JPG
9 IMG_2340.JPG


In [34]:
#Verify first two names
img1 = cv2.imread(str(working_images[0]))
img2 = cv2.imread(str(working_images[1]))

print("Image 1:", working_images[0].name)
print("Image 2:", working_images[1].name)
print("Identical:", np.array_equal(img1, img2))

Image 1: IMG_2331.JPG
Image 2: IMG_2332.JPG
Identical: False


# About images
Raw image files       : 100
Unique image contents : 100
Working images        : 100
Image 1               : IMG_2331.JPG
Image 2               : IMG_2332.JPG
Identical             : False